In [1]:
# 分析対象銘柄の証券コードをセット
from datetime import date
code = 9997
valuation_date = date.today()
# valuation_date = date(2022, 11, 18)

In [2]:
# 読み込みファイルパスの設定とimportしたいmoduleパス(pythonパス)の設定
from pathlib import Path
import os

CURRENT_DIR = Path(os.getcwd())
PJ_DIR = CURRENT_DIR.parent.parent
DATA_DIR = PJ_DIR / "data" 


# notebook内で利用するmoduleのimport
from wequant.data_processing import KessanPl, read_data, MeigaralistPl
import polars as pl
import plotly.io as pio
import plotly.graph_objects as go

DATEFORMAT2 = "%Y年%m月%d日"

In [20]:
# codeで指定した銘柄の四半期決算の業績推移のグラフ描画インスタンスfigを返す
# valuation_dateを指定された日で評価できるようにグラフを描画する(過去時点の評価が可能)
# def get_fig_quaterly_settlement_trend_barchart(code: int, valuation_date: date=date.today()) -> Figure:
# dataの読み込み
fp1 = DATA_DIR / "kessan.parquet"
df1 = read_data(fp1)

fp2 = DATA_DIR / "meigaralist.parquet"
df2 = read_data(fp2)

# valuation_dateで絞り込み
df1 = df1.filter(pl.col("announcement_date")<=valuation_date)

KPL = KessanPl(df1)

# xを決算期の表記に変更してcodeで指定した銘柄の四半期決算を抽出
KPL.with_columns_financtial_period()
df = KPL.df
df = df.filter(pl.col("code")==code)\
    .filter(pl.col("settlement_type")=="四")

#
# 売上高のbarchartをセット
#
# x軸のラベル用に列をカスタマイズして追加
df = df.with_columns([
    pl.col("quater").cast(pl.Utf8),
    pl.col("fy").cast(pl.Utf8),
    pl.col("fm").cast(pl.Utf8)
])
df = df.with_columns([
    (pl.col("fy")+pl.lit("-")+pl.col("fm")+pl.lit("-")+pl.col("quater")+pl.lit("Q")).alias("xlabels")
])

pandas_df = df.to_pandas()
sales_df = pandas_df[["xlabels", "sales"]]

# グラフ出力オプション
pio.renderers.default = 'iframe'

# 棒グラフのセット
graph_data = [
    go.Bar(
        x = sales_df["xlabels"],
        y = sales_df["sales"],
        marker = dict(color="skyblue"),
        name = "売上高"
    )
]
fig = go.Figure(graph_data)

# 利益率を右軸をy軸として、折れ線グラフでトレース
column_idx = 0
label_idx = 1
color_idx = 2
line_trace_cols_attrs = [
    ['operating_income', '営業利益', 'orange'],
    ['ordinary_profit', '経常利益', 'lightgreen'],
    ['final_profit', '純利益', 'purple']
]

for a in line_trace_cols_attrs:
    fig.add_trace(go.Scatter(
        x=pandas_df['xlabels'],
        y=pandas_df[a[column_idx]],
        mode='lines',
        name=a[label_idx],
        yaxis = 'y2',
        line=dict(color=a[color_idx], width=2)
    ))

# 年度の区切り線を引く
q4_df = pandas_df[pandas_df["quater"]=="4"]
vline_x_positions = q4_df.index
for q4x in vline_x_positions:
    xpos = int(q4x) + 0.5
    
    fig.add_vline(
        x=xpos,  # 棒の間に対応する位置
        line=dict(color='gray', width=2),
        annotation_text="",  # ラベル（任意）
        annotation_position="top"
    )

# レイアウトの設定
MPL = MeigaralistPl(df2)
company_name = MPL.get_name(code)
fig.update_layout(
    title=f'{company_name}({code})四半期業績推移({valuation_date.strftime(DATEFORMAT2)}時点)',
    xaxis=dict(title='年度'),
    yaxis=dict(title='売上高 (百万円)'),
    yaxis2=dict(
        title="利益(百万円)",
        overlaying="y", # 左のY軸に重ねる
        side="right"
    ),
    legend=dict(
        x=1.05,  # 凡例をグラフの外側に配置
        y=1,    # 上部に配置
        xanchor='left',  # 凡例の左端をx座標に揃える
        yanchor='top'    # 凡例の上端をy座標に揃える
    ),
    bargap=0.2  # 棒の間隔
)

fig.show()

In [11]:
vline_x_positions

Index([2, 6, 10, 14, 18, 22], dtype='int64')

In [9]:
q4_df

,code,settlement_date,settlement_type,announcement_date,sales,operating_income,ordinary_profit,final_profit,reviced_eps,dividend,quater,yearly_settlement_date,fy,fm,決算期,xlabels
2,9997,2019-03-31,四,2019-05-13,44093,3545,4001,3166,32.6,8.0,4,2019-03-31,2019,3,2019年3月期,2019-3-4Q
6,9997,2020-03-31,四,2020-05-13,42370,3037,2976,1268,13.1,7.2,4,2020-03-31,2020,3,2020年3月期,2020-3-4Q
10,9997,2021-03-31,四,2021-05-13,52354,4127,4385,2416,25.0,7.9,4,2021-03-31,2021,3,2021年3月期,2021-3-4Q
14,9997,2022-03-31,四,2022-05-13,51501,3609,3545,2459,25.4,7.0,4,2022-03-31,2022,3,2022年3月期,2022-3-4Q
18,9997,2023-03-31,四,2023-05-12,51922,3343,3623,2035,21.1,6.4,4,2023-03-31,2023,3,2023年3月期,2023-3-4Q
22,9997,2024-03-31,四,2024-05-13,52020,4062,4901,758,7.8,7.8,4,2024-03-31,2024,3,2024年3月期,2024-3-4Q


In [6]:
pandas_df

,code,settlement_date,settlement_type,announcement_date,sales,operating_income,ordinary_profit,final_profit,reviced_eps,dividend,quater,yearly_settlement_date,fy,fm,決算期,xlabels
0,9997,2018-09-30,四,2018-10-31,37182,1374,2101,1362,14.0,3.7,2,2019-03-31,2019,3,2019年3月期,2019-3-2Q
1,9997,2018-12-31,四,2019-01-31,54394,4610,4690,2949,30.3,8.5,3,2019-03-31,2019,3,2019年3月期,2019-3-3Q
2,9997,2019-03-31,四,2019-05-13,44093,3545,4001,3166,32.6,8.0,4,2019-03-31,2019,3,2019年3月期,2019-3-4Q
3,9997,2019-06-30,四,2019-07-31,46155,1951,1775,1113,11.5,4.2,1,2020-03-31,2020,3,2020年3月期,2020-3-1Q
4,9997,2019-09-30,四,2019-10-31,40067,1912,1521,565,5.8,4.8,2,2020-03-31,2020,3,2020年3月期,2020-3-2Q
5,9997,2019-12-31,四,2020-01-31,51356,3411,4093,2916,30.2,6.6,3,2020-03-31,2020,3,2020年3月期,2020-3-3Q
6,9997,2020-03-31,四,2020-05-13,42370,3037,2976,1268,13.1,7.2,4,2020-03-31,2020,3,2020年3月期,2020-3-4Q
7,9997,2020-06-30,四,2020-07-31,48534,2468,2669,1492,15.4,5.1,1,2021-03-31,2021,3,2021年3月期,2021-3-1Q
8,9997,2020-09-30,四,2020-10-30,42605,2097,2455,1431,14.8,4.9,2,2021-03-31,2021,3,2021年3月期,2021-3-2Q
9,9997,2020-12-31,四,2021-01-29,63006,7042,7363,5697,58.9,11.2,3,2021-03-31,2021,3,2021年3月期,2021-3-3Q
